In [3]:
import requests
def get_uniprot_batch(uniprot_ids, fields="accession,id,gene_names,protein_name,sequence"):
    ids_query = " OR ".join(f"accession:{uid}" for uid in uniprot_ids)
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": ids_query,
        "fields": fields,
        "format": "tsv"
    }
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    return response.text

ids = ["P05067", "P49768", "P49810"]  # APP, PSEN1, PSEN2
tsv_result = get_uniprot_batch(ids)

import pandas as pd
from io import StringIO
df = pd.read_csv(StringIO(tsv_result), sep="\t")


0    P49810
1    P49768
2    P05067
Name: Entry, dtype: object 0    PSEN2 AD4 PS2 PSNL2 STM2
1         PSEN1 AD3 PS1 PSNL1
2                  APP A4 AD1
Name: Gene Names, dtype: object


In [29]:
import requests
import json
import time
from pathlib import Path

CHUNK_SIZE = 1
OUTPUT_FILE = Path("protein_to_gene.json")
FAILED_FILE = Path("failed_ids.json")


def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def query_uniprot_batch(uniprot_ids, fields="accession,gene_names"):
    ids_query = " OR ".join(f"accession:{uid}" for uid in uniprot_ids)
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": ids_query,
        "fields": fields,
        "format": "tsv",
        "size": 500, 
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.text


def parse_tsv_to_dict(tsv_text):
    """Parses 'Entry\tGene Names' TSV into {accession: primary_gene_name}."""
    lines = tsv_text.strip().split("\n")
    result = {}
    for line in lines[1:]:
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        accession, gene_names = parts[0], parts[1]
        genes = gene_names.split() if gene_names.strip() else None
        result[accession] = genes
    return result


def load_existing(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}


def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


def build_protein_gene_dict(all_protein_ids, delay=0.5):
    protein_to_gene = load_existing(OUTPUT_FILE)
    failed_ids = load_existing(FAILED_FILE)


    remaining_ids = [pid for pid in all_protein_ids if pid not in protein_to_gene]
    print(f"{len(protein_to_gene)} already done, {len(remaining_ids)} remaining")

    chunks = list(chunk_list(remaining_ids, CHUNK_SIZE))

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i + 1}/{len(chunks)} ({len(chunk)} IDs)...")
        try:
            tsv_text = query_uniprot_batch(chunk)
            chunk_dict = parse_tsv_to_dict(tsv_text)
            protein_to_gene.update(chunk_dict)

            returned_ids = set(chunk_dict.keys())
            missing = set(chunk) - returned_ids
            for mid in missing:
                failed_ids[mid] = "no result returned"

        except requests.exceptions.RequestException as e:
            print(f"  Chunk {i + 1} failed: {e}")
            for pid in chunk:
                failed_ids[pid] = str(e)
        save_json(protein_to_gene, OUTPUT_FILE)
        save_json(failed_ids, FAILED_FILE)

        time.sleep(delay)

    print(f"Done. {len(protein_to_gene)} resolved, {len(failed_ids)} failed.")
    return protein_to_gene


if __name__ == "__main__":
    with open("Databases/uniprot_ids.txt") as f:
        all_ids = json.load(f)
        result = build_protein_gene_dict(all_ids)

27779 already done, 172 remaining
Processing chunk 1/172 (1 IDs)...
Processing chunk 2/172 (1 IDs)...
Processing chunk 3/172 (1 IDs)...
  Chunk 3 failed: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AENSP00000462298.1&fields=accession%2Cgene_names&format=tsv&size=500
Processing chunk 4/172 (1 IDs)...
Processing chunk 5/172 (1 IDs)...
Processing chunk 6/172 (1 IDs)...
Processing chunk 7/172 (1 IDs)...
Processing chunk 8/172 (1 IDs)...
  Chunk 8 failed: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AENSP00000493399.1&fields=accession%2Cgene_names&format=tsv&size=500
Processing chunk 9/172 (1 IDs)...
Processing chunk 10/172 (1 IDs)...
  Chunk 10 failed: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AENSP00000391869.3&fields=accession%2Cgene_names&format=tsv&size=500
Processing chunk 11/172 (1 IDs)...
Processing chunk 12/172 (1 IDs)...

None
